# w9_multianchor_5fold.ipynb — 多锚点五折 @4096（2 包 × 2048）

用户设计（2026-07-23）：**取消平均**。一个游戏用 N 个独立教师包表示，
**两个包都进画廊**作为独立条目；CE 对全部 `N × n_train` 个包做 softmax，
**自己游戏的 N 个包全部是正样本**（SupCon L_out = N 个 CE 取平均）；
包与包之间**不加任何直接拉近项** —— 它们只通过"同为一批学生视图的正
样本"而间接关联，因此各自保留侧面。检索按 **per-game max** 聚合。

动机：球面平均有损，且 pk 的 `pack-I` 把两包硬拉重合，与分包的初衷自相
矛盾（`pk4@2048` 精确持平基线 0.676 = 0.676 就是这个矛盾的后果）。多锚
点是绕过 128-d 容量瓶颈的正路——不是逼一个向量装更多（slot8 vs slot4
= +0.016, 3/5 = 噪声），而是给一个游戏多个向量各管一面。

**判据**（与 I-CE@4096 五折同折配对）：
- 检索 stripped hit@1 是否守住 **0.732 ± 0.013**
- **test_tag 能否突破语义带 0.711–0.717** ← 若多锚点真能绕开容量瓶颈，
  tag 应是第一个亮的信号（语义广度正是被平均杀死的东西）

预算对齐：`--anchor-cap 4096` = 2 包 × 2048 句，与 I-CE@4096 **总句数
相同**，显存同量级。读出同时报 **concat 探针**（方法自然读出，256-d）与
**pack0-only 探针**（128-d 同维对照），以分离"容量效应"与"多锚点效应"。
AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"

ARM = "wcle_ma2i2ce_icetf"          # 2 packs x 2048, both in the gallery
REF = "wcle_i2ce_icetf"             # the paired reference (single 4096 pack)
CAP, EPOCHS, N_FOLDS = 4096, 2000, 5
CKPT_EVERY = 50
os.makedirs(OUT_DIR, exist_ok=True)
print(f"multi-anchor 5-fold: {ARM}@{CAP} ({CAP // 2} sentences x 2 packs), "
      f"{EPOCHS}ep, paired against {REF}@{CAP}")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import os
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)

# PREBUILT ANCHOR PACK (required by multi-anchor): the worker builds anchors
# from the 2048-sentence POOL unless wscan_gal_rev_g{cap}.npz sits in
# DATA_DIR. A POOL-built pack realizes only ~2068 sentences, so at cap 4096
# each 2048-sentence sub-pack would cover the WHOLE pool and the two anchors
# would be ~99% identical (the arm silently degenerates to single-anchor).
# Symlinked, not copied: /dev/shm already carries the 140 GB full pool.
_pack = f"wscan_gal_rev_g{CAP}.npz"
_ps, _pd = src / _pack, dst / _pack
assert _ps.exists(), (
    f"{_pack} missing from {DATA_SRC}. Multi-anchor needs full-corpus "
    f"anchors; without it the packs collapse together. Fetch it from the "
    f"bucket (fusion_cache_w9/{_pack}, 16.9 GB) first.")
if not _pd.exists():
    try:
        os.symlink(_ps, _pd)
        print(f"linked {_pack} -> {_ps}")
    except OSError:
        print(f"symlink unavailable; copying {_pack} ({_ps.stat().st_size/1e9:.1f} GB) ...",
              flush=True)
        shutil.copyfile(_ps, _pd)
print("corpus in RAM:", DATA_DIR, f"| anchor pack: {_pd}")

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM probe BEFORE committing the queue: 3 smoke steps write the peak and
# exit. Multi-anchor encodes 2 x (n_train x 2048) with gradient per step --
# the same total sentence count as I-CE@4096, so the expectation is ~45-50G.
import os, subprocess, sys, tempfile
from pathlib import Path

tf = Path(tempfile.gettempdir()) / "ma2_vram.txt"
tf.unlink(missing_ok=True)
gpus = J.detect_gpus()
cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
       os.path.join(tempfile.gettempdir(), "ma2_probe"), "--repo", REPO,
       "--arm", ARM, "--fold", "0", "--n-folds", str(N_FOLDS),
       "--anchor-cap", str(CAP), "--epochs", "1",
       "--full-pool", "--full-pool-path", FULL_POOL_PATH,
       "--measure-vram", str(tf)]
print("probing VRAM ...", flush=True)
subprocess.run(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0],
                             PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
if tf.exists():
    peak = int(tf.read_text())
    print(f"[measure-vram] multi-anchor peak = {peak / 2**30:.1f} GiB")
    assert peak < 75 * 2**30, "peak too close to the 80G ceiling -- rethink before queueing"
else:
    raise SystemExit(
        "VRAM probe produced no peak -- it CRASHED. Read the probe output above "
        "before queueing: a probe that dies (OOM, degenerate-pack assert, scope "
        "error) means every one of the 5 folds would die the same way. The first "
        "live run proceeded past this point and burned 5 cards on a run whose "
        "packs were 99% identical.")


In [ ]:
# Five folds, one per GPU round-robin. Done-marker = tower npz at EPOCHS.
import os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"; cdir.mkdir(parents=True, exist_ok=True)
logd = Path(OUT_DIR) / "logs"; logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()

todo = []
for k in range(N_FOLDS):
    nm = f"w9cv_{ARM}_fold{k}_g{CAP}"
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
        print(f"[skip] {nm} done"); continue
    todo.append((k, nm))
print(f"{len(todo)} fold(s) to run on {len(gpus)} GPU(s)")

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails, retried = [], set()

def run_one(g, k, nm):
    if not J.try_claim(cdir, nm):
        print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", ARM, "--fold", str(k),
           "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] start {nm}", flush=True)
    t0 = time.time()
    with open(logd / f"{nm}.log", "w") as fh:
        pr = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=g,
                                     PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
    ok = pr.returncode == 0
    if not ok:
        (cdir / f"{nm}.claim").unlink(missing_ok=True)
        if nm not in retried:
            retried.add(nm); print(f"[retry] {nm} once", flush=True)
            return run_one(g, k, nm)
        fails.append(nm)
    print(f"[gpu{g}] {'ok' if ok else 'FAIL'} {nm} [{(time.time()-t0)/3600:.1f}h]",
          flush=True)

ths = [threading.Thread(target=run_one, args=(gpus[i % len(gpus)], k, nm))
       for i, (k, nm) in enumerate(todo)]
for i, t in enumerate(ths):
    if i: time.sleep(60)
    t.start()
for t in ths: t.join()
stop_evt.set()
print(f"done; {len(fails)} failed: {fails}")


In [ ]:
# VERDICT: multi-anchor vs the two reigning champions.
#   name / stripped  -> I-CE@4096 (0.947 / 0.732)
#   test-set TAG F1  -> BYOL      (0.717)   <- the tag champion
# The tag number is read from a MEAN-POOLED 128-d vector -- the same structure
# and width every other arm feeds its probe -- so the BYOL comparison is
# dimension-fair. Retrieval uses per-game MAX over packs, which is where
# multi-anchor is supposed to pay off.
import json
import numpy as np
from pathlib import Path

ICE_NEU, ICE_NON, BYOL_TAG = 0.947, 0.732, 0.717


def rows(arm):
    out = {}
    for k in range(N_FOLDS):
        p = Path(OUT_DIR) / f"zsbest_w9cv_{arm}_fold{k}_g{CAP}_fp.json"
        if p.exists():
            out[k] = json.loads(p.read_text())
    return out


ma, ref = rows(ARM), rows(REF)


def show(lab, r):
    if not r:
        print(f"{lab:24s} (pending)")
        return
    M = lambda f: np.mean([x[f] for x in r.values()])
    S = lambda f: np.std([x[f] for x in r.values()])
    print(f"{lab:24s} n={len(r)}  neu {M('nm_neutral'):.3f}+-{S('nm_neutral'):.3f}  "
          f"stripped {M('nm_noname'):.3f}+-{S('nm_noname'):.3f}  "
          f"tag {M('test_tag'):.3f}+-{S('test_tag'):.3f}")


show("multi-anchor (2 packs)", ma)
show("I-CE (single 4096)", ref)

if ma:
    pc = [x.get("pack_cos") for x in ma.values() if x.get("pack_cos") is not None]
    if pc:
        m = float(np.mean(pc))
        print(f"\npack_cos (mean cosine between the two packs) = {m:.3f}")
        print("  -> " + ("DEGENERATE: packs collapsed; the arm is effectively "
                         "single-anchor and any comparison below is void"
                         if m > 0.98 else
                         "packs stayed distinct; multi-anchor is doing real work"))
    mn = float(np.mean([x["nm_neutral"] for x in ma.values()]))
    ms = float(np.mean([x["nm_noname"] for x in ma.values()]))
    mt = float(np.mean([x["test_tag"] for x in ma.values()]))
    print("\n=== HEAD-TO-HEAD vs the champions ===")
    for lab, got, tgt, who in (("name hit@1", mn, ICE_NEU, "I-CE"),
                               ("stripped hit@1", ms, ICE_NON, "I-CE"),
                               ("test-set TAG F1", mt, BYOL_TAG, "BYOL")):
        d = got - tgt
        v = "BEATS" if d > 0.005 else ("matches" if d > -0.005 else "loses to")
        print(f"  {lab:16s} {got:.3f}  vs {who} {tgt:.3f}   {d:+.3f}  -> {v} {who}")
    if ms >= ICE_NON - 0.005 and mt >= BYOL_TAG - 0.005:
        print("\n  *** BOTH CROWNS: retrieval at I-CE level AND tag at BYOL level ***")

ks = sorted(set(ma) & set(ref))
if ks:
    print(f"\nPAIRED vs I-CE over {len(ks)} folds:")
    for f, lab in (("nm_noname", "stripped hit@1"), ("nm_neutral", "name hit@1"),
                   ("test_tag", "test-set TAG F1")):
        d = [ma[k][f] - ref[k][f] for k in ks]
        print(f"  {lab:16s} d {np.mean(d):+.3f}  wins {sum(1 for x in d if x > 0)}/{len(d)}  "
              f"per-fold {[round(x, 3) for x in d]}")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")